# Guesser prepared-data audit

This notebook audits **labels and split manifests for production, medium, or small** without a trained model. It reads the selected profile's contract from the data package and checks the label registry, per-class split counts, manifest row shape, and drawing-ID / trajectory-hash disjointness. The uniqueness check is disk-backed so it does not load all IDs into notebook memory.

It does not calculate recognition metrics, inspect the indexed JSONL payload bytes, or render images. The data package's `guesser-data audit` checks the prepared payload; model predictions are required for accuracy or calibration. Keep the held-out test package sealed until the model and evaluation protocol are frozen.

Run from the repository root with `uv run --project models/guesser/evaluation jupyter lab models/guesser/evaluation/notebooks`. Prepare the separate data environment first, when no data job is active. Profile discovery uses `uv run --frozen --no-sync` with Python 3.13 and never starts downloads or syncs dependencies. Re-run the audit cell after the data pipeline finishes.

In [ ]:
# Reload our generated module after its writer cell runs, without restarting the kernel.
%load_ext autoreload
%autoreload 2

## Options and dataset location

| Option | Default | Effect |
|---|---|---|
| `PROFILE` | `None` | Use the data CLI default (currently production), or select `production`, `medium`, or `small`. Smoke is not trainable and is rejected. Expected counts are read from `settings.py`, not copied here. |
| `DATA_ROOT` | `None` | Use the selected profile's directory. An explicit path overrides `MTMAI_GUESSER_DATA_ROOT`, which overrides the default. Both overrides name the directory **containing `prepared/`, `labels.json`, and `dataset-summary.json`**, not its parent as in notebook 01. Relative overrides use the kernel working directory. |

The audit needs all three split manifests and rejects a summary whose profile differs from the selection. An extracted trainer ZIP has no test manifest and cannot pass this full split-isolation audit by itself. Keep the held-out package sealed until the model and evaluation protocol are frozen; do not extract it just to run this notebook early. Medium/small are prefix samples, not controlled subsets of production.

In [ ]:
# Resolve the helper path from this checkout so the notebook works from different cwd values.
from guesser_evaluation.notebook_tools import module_source_path

workflow_source_path = str(module_source_path('dataset_audit_workflow.py'))

# None delegates to the source defaults; explicit paths name a profile directory.
PROFILE = None
DATA_ROOT = None
print(f'Workflow source: {workflow_source_path}')

In [ ]:
%%writefile {workflow_source_path}
"""Notebook helpers for locating, auditing, and plotting Guesser data."""

from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import Any


def data_profile(profile: str | None = None) -> dict[str, Any]:
    """Read the selected profile contract in the data project's own environment."""
    from .notebook_tools import module_source_path

    source_path = module_source_path("dataset_audit_workflow.py")
    data_project = source_path.parents[2].parent / "data"
    # Query settings without syncing or importing Python 3.13 code into this kernel.
    query = """
import json, sys
from guesser_data.cli import parser
from guesser_data.settings import PACKAGE_ROOT, Settings
defaults = parser().parse_args(["download"])
settings = Settings(profile=sys.argv[1] or defaults.profile)
if not settings.is_trainable:
    raise ValueError("Audit requires a trainable profile, not smoke")
print(json.dumps({
    "profile": settings.profile, "data_root": str(settings.base),
    "counts": {split: settings.spec[split] for split in ("train", "validation", "test")},
    "class_count": len((PACKAGE_ROOT / "src/resources/categories.txt").read_text().splitlines()),
    "sampling_method": settings.download_method,
}))
"""
    result = subprocess.run(
        [
            "uv",
            "run",
            "--project",
            str(data_project),
            "--frozen",
            "--no-sync",
            "python",
            "-c",
            query,
            "" if profile is None else profile,
        ],
        cwd=data_project.parents[2],
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(result.stdout)


def profile_data_root(profile: str | None = None) -> Path:
    """Resolve a profile directory, preserving the legacy environment override."""
    contract = data_profile(profile)
    return Path(os.environ.get("MTMAI_GUESSER_DATA_ROOT", contract["data_root"])).expanduser()


def production_data_root() -> Path:
    """Keep the original production-only path helper compatible."""
    return profile_data_root("production")


def audit_dataset(
    data_root: str | Path | None = None, *, profile: str | None = None
) -> dict[str, Any] | None:
    """Audit a selected training profile, returning None while preparation is pending."""
    from .dataset_audit import audit_prepared_dataset

    contract = data_profile(profile)
    selected = contract["profile"] if profile is None else profile
    # Explicit paths and the legacy environment variable name a profile directory.
    configured = (
        data_root
        if data_root is not None
        else os.environ.get("MTMAI_GUESSER_DATA_ROOT", contract["data_root"])
    )
    root = Path(configured).expanduser()
    prepared_path = root / "prepared"
    # A missing folder means preparation is pending, not that the audit passed.
    if not prepared_path.is_dir():
        return None
    summary = json.loads((root / "dataset-summary.json").read_text(encoding="utf-8"))
    if summary.get("profile") != selected:
        raise ValueError(f"Dataset profile does not match selected profile {selected!r}")
    return audit_prepared_dataset(
        prepared_path,
        # Counts and registry size come from the data source, not duplicated constants.
        expected_class_count=contract["class_count"],
        expected_per_class=contract["counts"],
    )


def audit_production_dataset(data_root: str | Path | None = None) -> dict[str, Any] | None:
    """Preserve the existing production-only public entry point."""
    return audit_dataset(data_root, profile="production")


def plot_class_balance(report: dict[str, Any]):
    """Return per-split totals and a class-count figure from a passed audit."""
    import matplotlib.pyplot as plt
    import pandas as pd

    counts = pd.DataFrame(report["counts_by_class"])
    # Sum the same audited rows shown in the plot for a quick total check.
    totals = counts[["train", "validation", "test"]].sum()
    # Stack partitions so class balance can be compared without hiding any split.
    figure, axis = plt.subplots(figsize=(18, 5))
    counts.set_index("source_name")[["train", "validation", "test"]].plot(
        kind="bar", stacked=True, width=0.9, ax=axis
    )
    axis.set_title("Original drawings by Quick, Draw! class and split")
    axis.set_xlabel("Class")
    axis.set_ylabel("Original drawings")
    figure.tight_layout()
    return totals, figure


## Audit labels, counts, and split isolation

The selected profile's expected counts, label count, and sampling method are displayed below. The function reads Parquet manifests in batches and uses a temporary SQLite index to detect duplicate identities and trajectories across all splits. Missing preparation returns a pending state, not a passing audit. This reads test manifests when they are present in the selected dataset, but does not open ZIPs or read test drawing payloads.

In [ ]:
# Refresh discovery after %%writefile creates the reusable workflow module.
import importlib
importlib.invalidate_caches()
from guesser_evaluation.dataset_audit_workflow import audit_dataset, data_profile, profile_data_root

print('Selected profile contract:', data_profile(PROFILE))
print('Dataset directory:', DATA_ROOT if DATA_ROOT is not None else profile_data_root(PROFILE))
audit_report = audit_dataset(DATA_ROOT, profile=PROFILE)
if audit_report is None:
    # Missing data means acquisition/preparation is pending, not that validation passed.
    audit_report = None
    print('Prepared data is not available yet. Re-run this cell after download and prepare finish.')
audit_report

## Inspect class balance

The chart visualizes counts from the manifests after the audit passes. It is a dataset-structure check, not model performance. A failed audit raises an error above instead of silently omitting rows.

In [ ]:
from IPython.display import display

if audit_report is not None:
    # Only visualize class balance after every manifest check has passed.
    from guesser_evaluation.dataset_audit_workflow import plot_class_balance

    split_totals, class_balance_figure = plot_class_balance(audit_report)
    display(split_totals.to_frame(name='originals'))
    class_balance_figure
else:
    print('No prepared manifests to summarize yet.')

## Limits and next step

A passing result establishes that the label/split manifests satisfy these checks; it does **not** establish that each indexed JSONL byte range is intact or that preprocessing looks correct. Run the data package audit for payload integrity. When checkpoints arrive, extend evaluation with the fixed ten-stage protocol, keeping blank/pass opportunities in the denominator and using validation for model selection and calibration. Do not tune against held-out test results.

The writer cell is the source of truth for `src/guesser_evaluation/dataset_audit_workflow.py`; execute it to regenerate the importable module. The independent manifest auditor lives in `src/guesser_evaluation/dataset_audit.py`. Keep both modules covered by tests.